# Capstone Project: Personal Expense Tracker

This mini capstone pulls together everything covered in the Series module so
far. You have been logging every expense for over a year. The log lives in
`expenses.csv` (1,236 transactions) and, like all real logs, it is messy:

- Amounts are **strings** with a leading `₦`, typed by hand
- Category labels have **inconsistent casing and stray spaces**
- Some amounts were **never recorded** and come in blank
- The **date** is the natural row identifier

Your job is to turn this raw log into real answers using Series tools only.

**The project will only consist of concepts learned so far**:

- Reading a CSV column into a Series; `.values`, `.index`, `.name`, `.dtype`,
  `.size`
- `.astype()` type conversion
- Custom indices, `[]`, `.iloc`, `.loc`, `reset_index(drop=True)`
- Filtering with logical tests, `.isin()`, `~`, and masks
- `.sort_values()` and `.sort_index()`
- Numeric operations (operators, methods, `fill_value`)
- Text operations via the `.str` accessor
- Aggregations: `.sum`, `.mean`, `.median`, `.count`, `.min`, `.max`,
  `.std`, `.quantile`
- Categorical aggregation: `.unique`, `.nunique`, `.value_counts`
- Missing data: `.isna`, `.dropna`, `.fillna`


In [1]:
import numpy as np
import pandas as pd

## Loading the Data

Read the CSV. We pull two columns out as separate Series, each indexed by the
transaction date. Working with two aligned Series like this is a preview of
what a DataFrame does for us later.

The `date` column is set as the index, so a single transaction is addressable
by its date and the two Series stay aligned.

In [2]:
raw = pd.read_csv("expenses.csv", index_col="date")
raw.head()

,amount,category
date,,
2023-01-01,₦800.38,coffee
2023-01-02,₦1900.35,Groceries
2023-01-03,₦3700.70,dining
2023-01-03,₦800.70,Coffee
2023-01-03,₦2600.15,transport


The amount column, still raw `₦`-prefixed strings (note the blanks read in as
missing values).

In [3]:
spend = pd.Series(raw["amount"].to_numpy(), index=raw.index, name="amount")
spend.head(10)

date
2023-01-01     ₦800.38
2023-01-02    ₦1900.35
2023-01-03    ₦3700.70
2023-01-03     ₦800.70
2023-01-03    ₦2600.15
2023-01-03     ₦400.35
2023-01-04    ₦6700.33
2023-01-04    ₦2500.74
2023-01-04    ₦5000.36
2023-01-05    ₦5900.71
Name: amount, dtype: object

The category column, with all its messy casing and spacing.

In [4]:
category = pd.Series(raw["category"].to_numpy(), index=raw.index, name="category")
category.head(10)

date
2023-01-01        coffee
2023-01-02    Groceries 
2023-01-03        dining
2023-01-03        Coffee
2023-01-03    transport 
2023-01-03        Coffee
2023-01-04     groceries
2023-01-04       Dining 
2023-01-04       Health 
2023-01-05     groceries
Name: category, dtype: object

A quick sense of scale before we start.

In [5]:
spend.size

1236

In [6]:
spend.dtype

dtype('O')

## Challenge 1: Clean the Amounts

The amounts are unusable for maths right now - they are Naira-prefixed
strings with some missing values.

1. Strip the `₦` from every amount and convert the Series to a float.
2. Confirm how many amounts were missing.
3. Fill the missing amounts with `0`, since a blank means you forgot to log the
   cost and we will treat it as no recorded spend.

Store the cleaned result back in `spend`.

### Solution 1

Strip the naira sign and cast to float. The missing values stay as NaN
through the conversion.

In [7]:
spend = spend.str.strip("₦").astype(float)
spend.head()

date
2023-01-01     800.38
2023-01-02    1900.35
2023-01-03    3700.70
2023-01-03     800.70
2023-01-03    2600.15
Name: amount, dtype: float64

Count the missing amounts.

In [8]:
spend.isna().sum()

30

As a proportion of all transactions.

In [9]:
spend.isna().mean()

0.024271844660194174

Fill the missing amounts with 0.

In [10]:
spend = spend.fillna(0)
spend.isna().sum()

0

## Challenge 2: Clean the Categories

The category labels are inconsistent: mixed casing (`coffee`, `Coffee`) and
stray spaces (`"Groceries "`, `" groceries"`). Left as-is, the same category
would be counted as several different ones.

1. First, confirm how many *distinct raw labels* exist before cleaning.
2. Strip stray spaces from both ends of every label.
3. Convert every label to a single consistent case.
4. Confirm how many distinct categories you actually have, and list them.

### Solution 2

The raw label count is inflated by casing and spacing duplicates.

In [11]:
category.nunique()

27

Strip surrounding spaces, then lowercase everything so the variants collapse.

In [12]:
category = category.str.strip().str.lower()
category.head()

date
2023-01-01       coffee
2023-01-02    groceries
2023-01-03       dining
2023-01-03       coffee
2023-01-03    transport
Name: category, dtype: object

The true number of distinct categories, far fewer than the raw count.

In [13]:
category.nunique()

8

The distinct category labels.

In [14]:
category.unique()

array(['coffee', 'groceries', 'dining', 'transport', 'health',
       'utilities', 'shopping', 'rent'], dtype=object)

## Challenge 3: The Headline Numbers

Answer each question with a single aggregation on `spend`:

1. Total spent across the whole log.
2. Average spend per transaction.
3. The single largest transaction.
4. The typical (median) transaction, which ignores the big one-off costs.

### Solution 3

Total spend.

In [15]:
spend.sum()

4108045.66

Average spend per transaction.

In [16]:
spend.mean()

3323.6615372168285

Largest single transaction.

In [17]:
spend.max()

120000.0

Median transaction. It sits well below the mean, telling you a small number of
large costs (rent) pull the average up.

In [18]:
spend.median()

1900.32

## Challenge 4: Where Did the Money Go?

Get a feel for spending habits before totalling anything.

1. How often does each category appear in the log (raw counts)?
2. What share of your transactions does each category represent (as a
   proportion)?

### Solution 4

Raw counts per category - coffee and groceries dominate by frequency.

In [19]:
category.value_counts()

category
coffee       358
groceries    276
transport    264
dining       148
shopping      86
utilities     55
health        44
rent           5
Name: count, dtype: int64

The same breakdown as proportions, which is reportable no matter how many
transactions you logged.

In [20]:
category.value_counts(normalize=True)

category
coffee       0.289644
groceries    0.223301
transport    0.213592
dining       0.119741
shopping     0.069579
utilities    0.044498
health       0.035599
rent         0.004045
Name: proportion, dtype: float64

## Challenge 5: Spend Per Category

Counts tell you how *often* you buy something, not how much it *costs*. Because
`spend` and `category` share the same index, a Boolean test on `category`
filters `spend` directly.

1. Total spent on groceries.
2. Total spent on coffee.
3. Total spent on everything that is **not** rent (use the tilde).

### Solution 5

Total grocery spend.

In [21]:
spend.loc[category == "groceries"].sum()

1294231.25

Total coffee spend - frequent, but small amounts each.

In [22]:
spend.loc[category == "coffee"].sum()

214355.88

Everything except rent, using the tilde to invert the test.

In [23]:
spend.loc[~(category == "rent")].sum()

3628045.66

## Challenge 6: The Discretionary Total

You class coffee, transport, dining, and shopping as the spending you can
actually influence day to day. Rent, utilities, and health are effectively
fixed.

1. Use `.isin()` to total the spend across just those four categories.
2. What proportion of all spending did that discretionary spending make up?

### Solution 6

Total discretionary spend.

In [24]:
discretionary = ["coffee", "transport", "dining", "shopping"]
discretionary_total = spend.loc[category.isin(discretionary)].sum()
discretionary_total

1637268.08

Its share of the whole log.

In [25]:
discretionary_total / spend.sum()

0.39855157792963775

## Challenge 7: Sorting to Find the Extremes

1. Sort the spend Series chronologically by its date index.
2. Find the single largest transaction and the date it landed on. (Hint: sort
   by value, descending, and look at the top row.)
3. List the 5 largest transactions.

### Solution 7

Sorted chronologically by the date index.

In [26]:
spend.sort_index().head()

date
2023-01-01     800.38
2023-01-02    1900.35
2023-01-03    3700.70
2023-01-03     800.70
2023-01-03    2600.15
Name: amount, dtype: float64

Sorted by value, largest first. The top row's index label is the date of the
biggest hit.

In [27]:
spend.sort_values(ascending=False).head()

date
2024-01-02    120000.00
2023-09-02    120000.00
2023-12-03    120000.00
2023-10-01    120000.00
2023-09-25     17300.16
Name: amount, dtype: float64

The single largest amount, by positional access on the sorted Series.

In [28]:
spend.sort_values(ascending=False).iloc[0]

120000.0

The 5 largest transactions.

In [29]:
spend.sort_values(ascending=False).iloc[:5]

date
2024-01-02    120000.00
2023-09-02    120000.00
2023-12-03    120000.00
2023-10-01    120000.00
2023-09-25     17300.16
Name: amount, dtype: float64

## Challenge 8: Filtering by Month

The date index is text in the form `YYYY-MM-DD`, so the month sits in
positions 5:7. You can slice the index strings to filter by month.

1. Total spend in March (month "03") of any year.
2. How many transactions happened in the first quarter (months 01, 02, 03)?

### Solution 8

Build a month mask by slicing the index strings, then total March spend.

In [30]:
months = pd.Series(spend.index).str[5:7].to_numpy()
spend.loc[months == "03"].sum()

264344.17000000004

Count of first-quarter transactions using `.isin()` on the month slice.

In [31]:
q1_mask = pd.Series(spend.index).str[5:7].isin(["01", "02", "03"]).to_numpy()
spend.loc[q1_mask].count()

376

## Challenge 9: Budget Check

Your rule of thumb is that no everyday transaction should exceed `₦100`. Audit
the log against that. (Rent will obviously break it - that is expected.)

1. Build a mask for transactions over `₦100` and list the first few.
2. How many transactions broke the rule?
3. By how much, in total, did those transactions exceed the `₦100` limit?
   (Hint: subtract 100 from each over-limit amount, then sum.)

### Solution 9

Transactions over the limit.

In [32]:
over_limit = spend.loc[spend > 100]
over_limit.head()

date
2023-01-01     800.38
2023-01-02    1900.35
2023-01-03    3700.70
2023-01-03     800.70
2023-01-03    2600.15
Name: amount, dtype: float64

Count of rule-breakers.

In [33]:
over_limit.count()

1206

Total overshoot beyond the 100 limit.

In [34]:
(over_limit - 100).sum()

3987445.66

## Challenge 10: How Spread Out Is Your Spending?

A single average hides how erratic spending is. Quantify the spread.

1. The standard deviation of transaction amounts.
2. The 25th, 50th, and 75th percentiles in one call.
3. The interquartile range (75th minus 25th), the band most spending falls in.

### Solution 10

Standard deviation of the amounts.

In [35]:
spend.std()

7238.577601289067

The three quartiles in one call.

In [36]:
quartiles = spend.quantile([0.25, 0.5, 0.75])
quartiles

0.25     700.5875
0.50    1900.3200
0.75    4700.1750
Name: amount, dtype: float64

The interquartile range, accessed by label from the quartile result.

In [37]:
quartiles.loc[0.75] - quartiles.loc[0.25]

3999.5875

## Challenge 11: Project Next Year

Costs are rising. Build a rough forecast.

1. Apply a flat 8% increase to every transaction to model inflation.
2. On top of that, add a fixed `₦2` surcharge to each transaction.
3. Compare the projected total against the actual total, expressed as a
   percentage increase.

### Solution 11

Apply the 8% increase and the flat surcharge.

In [38]:
projected = spend * 1.08 + 2
projected.head()

date
2023-01-01     866.4104
2023-01-02    2054.3780
2023-01-03    3998.7560
2023-01-03     866.7560
2023-01-03    2810.1620
Name: amount, dtype: float64

Actual total versus projected total.

In [39]:
spend.sum()

4108045.66

In [40]:
projected.sum()

4439161.3127999995

The percentage increase from actual to projected.

In [41]:
(projected.sum() - spend.sum()) / spend.sum()

0.08060174598935674

## Wrap-Up

Starting from a raw 1,236-row CSV, you cleaned messy real-world input (string
amounts and untidy labels), handled missing data, summarised the log with
aggregations, sliced spending by category using masks and `.isin()`, filtered
by month off the date index, audited against a budget rule, measured the
spread, and built a simple forecast - all with Series tools alone.

This is exactly the kind of work that becomes far smoother once these columns
live together in a single DataFrame, which is where the course heads next.